# Line Quality: Jitter, Packet Loss, and Level

Simulate a minute of call audio crossing a real network, then decide how long to wait for packets and what to play when they never come. Predict each result before running the cell.

In [ ]:
import sys
sys.path.insert(0, '../code')
import numpy as np
import main

## Same loss rate, different bursts

Both channels lose about 3% of packets. How long are their bursts?

In [ ]:
for name, p, r in (('random', 0.03, 0.97), ('bursty', 0.0062, 0.2)):
    lost = main.gilbert_elliott(50_000, p, r, seed=0)
    print(f"{name:<7} loss {100 * lost.mean():.2f}%  mean burst {main.burst_lengths(lost).mean():.2f} packets")

## Fixed versus adaptive jitter buffer

Remove the route change (`step_at=None`) and run again. Does the adaptive buffer still win?

In [ ]:
n = 3000
lost = main.gilbert_elliott(n, 0.01, 0.4, seed=1)
delays = main.network_delays(n, seed=2, step_at=1800, step_ms=60.0)
for buffer_ms in (40, 80, 120):
    jb = main.fixed_jitter_buffer(delays, buffer_ms, lost)
    print(f"fixed {buffer_ms:>3} ms: late {100 * jb['late_rate']:5.2f}%  delay {jb['mouth_to_ear_ms']:.0f} ms")
ajb = main.adaptive_jitter_buffer(delays, lost)
print(f"adaptive  : late {100 * ajb['late_rate']:5.2f}%  delay {ajb['mouth_to_ear_ms']:.0f} ms")

## Concealment and how to score it

Compare a waveform metric with a spectral one. Which ranking matches what you would hear?

In [ ]:
x = main.voiced_signal()
missing = np.zeros(50, dtype=bool)
missing[[10, 25, 26, 40, 41, 42]] = True
received = x * ~main.frames_to_samples(missing, 160)
for method in ('zero', 'repeat', 'pitch'):
    y = main.conceal(received, missing, 160, method)
    print(f"{method:<6} segSNR {main.segmental_snr_db(x, y):5.1f} dB   LSD {main.log_spectral_distance(x, y, missing):5.1f} dB")

## Level

Turn the noise gate off (`gate_dbfs=-120`) and look at what the AGC does to the silent first half.

In [ ]:
quiet = np.concatenate([0.001 * np.random.default_rng(0).standard_normal(4000), 0.03 * main.voiced_signal()])
for gate in (-50.0, -120.0):
    y = main.agc(quiet, gate_dbfs=gate)
    print(f"gate {gate:>6} dBFS: pause {main.dbfs(y[:4000]):6.1f} dBFS, speech {main.dbfs(y[8000:]):6.1f} dBFS")

## E-model

How much mouth-to-ear delay can a G.711 call with 1% loss afford before R drops below 80?

In [ ]:
for d in range(100, 301, 25):
    r = main.r_factor(d, 0.01)
    print(f"{d:>3} ms  R {r:5.1f}  MOS {main.mos_from_r(r):.2f}")